In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 56.4 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install pymupdf
#detailed_word_counts,section_summary,company_Summary,section_presence
import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-companies-annual-reports-fy-202223/Annual_Report_23/annual_reports_2023"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_Ann_22_23.csv", index=False)
    section_stats.to_csv("section_summary_Ann_22_23.csv")
    company_stats.to_csv("company_summary_Ann_22_23.csv")
    section_presence.to_csv("section_presence_Ann_22_23.csv")

    print("EDA Completed")

Total reports: 1394
MuPDF error: library error: FT_New_Memory_Face(UJALIE+Helvetica-Bold): unknown file format

MuPDF error: library error: FT_New_Memory_Face(UJALIE+Helvetica): unknown file format

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearance stream for  widgets

MuPDF error: argument error: cannot create appearan

In [ ]:
#sector_wise_companies

import os

import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-companies-annual-reports-fy-202223/Annual_Report_23/annual_reports_2023"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_Ann_22_23.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_Ann_22_23.csv", index=False)

    print("Sector files created successfully")

Total reports: 1394


In [ ]:
import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2024-25/Indian_BRSR_Reports_2024-25/BRSR_Reports_2024_25"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_brsr_24_25.csv", index=False)
    section_stats.to_csv("section_summary_brsr_24_25.csv")
    company_stats.to_csv("company_summary_brsr_24_25.csv")
    section_presence.to_csv("section_presence_brsr_24_25.csv")

    print("EDA Completed")

In [ ]:
import os
import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2024-25/Indian_BRSR_Reports_2024-25/BRSR_Reports_2024_25"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_brsr_24_25.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_brsr_24_25.csv", index=False)

    print("Sector files created successfully")

In [ ]:
import os
import fitz
import pandas as pd
import nltk
import re
from multiprocessing import Pool, cpu_count
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2025-26/brsr reports 2025-26/files"

sections = [
    "general disclosures",
    "management and process disclosures",
    "principle wise performance",
    "environment",
    "social",
    "governance"
]


def process_pdf(file):

    company = file.replace(".pdf", "")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read only first 60 pages (BRSR usually early in report)
        for page in doc[:60]:
            text += page.get_text()

    except:
        return None

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    total_words = len(words)

    words_no_stop = [w for w in words if w not in stop_words]

    total_words_no_stop = len(words_no_stop)

    results = []

    results.append({
        "Company": company,
        "Section": "TOTAL_REPORT",
        "Word_Count": total_words,
        "Word_Count_No_Stopwords": total_words_no_stop
    })

    for section in sections:

        if section in text:

            section_text = text.split(section, 1)[1]

            words = section_text.split()

            wc = len(words)

            words_no_stop = [w for w in words if w not in stop_words]

            wc_no_stop = len(words_no_stop)

            results.append({
                "Company": company,
                "Section": section,
                "Word_Count": wc,
                "Word_Count_No_Stopwords": wc_no_stop
            })

    return results


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(process_pdf, files)

    data = []

    for r in results:
        if r:
            data.extend(r)

    df = pd.DataFrame(data)

    section_stats = df.groupby("Section")[["Word_Count","Word_Count_No_Stopwords"]].sum()

    company_stats = df[df["Section"]=="TOTAL_REPORT"][["Company","Word_Count","Word_Count_No_Stopwords"]]

    section_presence = pd.crosstab(df["Company"], df["Section"])

    df.to_csv("detailed_word_counts_brsr_25_26.csv", index=False)
    section_stats.to_csv("section_summary_brsr_25_26.csv")
    company_stats.to_csv("company_summary_brsr_25_26.csv")
    section_presence.to_csv("section_presence_brsr_25_26.csv")

    print("EDA Completed")

In [ ]:
import os
import fitz
import pandas as pd
import re
from multiprocessing import Pool, cpu_count

reports_folder = "/kaggle/input/datasets/meghanakadari/indian-listed-companies-brsr-reports-fy-202324/Indian_BRSR_Reports_2025-26/brsr reports 2025-26/files"

# Sector keyword dictionary
sector_keywords = {
    "Information Technology": ["software", "technology", "it services", "digital"],
    "Financial Services": ["bank", "finance", "financial services", "nbfc", "insurance"],
    "Energy": ["oil", "gas", "energy", "petroleum"],
    "Healthcare": ["pharma", "pharmaceutical", "healthcare", "biotech"],
    "Consumer Goods": ["consumer goods", "fmcg", "retail", "food"],
    "Industrials": ["engineering", "industrial", "manufacturing"],
    "Materials": ["chemicals", "cement", "steel", "materials"],
    "Real Estate": ["real estate", "construction", "property"],
    "Telecommunications": ["telecom", "communication", "network"]
}


def detect_sector(file):

    company = file.replace(".pdf","")
    path = os.path.join(reports_folder, file)

    text = ""

    try:
        doc = fitz.open(path)

        # read first 15 pages only (sector usually mentioned early)
        for page in doc[:15]:
            text += page.get_text()

    except:
        return {"Company": company, "Sector": "Unknown"}

    text = text.lower()

    detected_sector = "Unknown"

    for sector, keywords in sector_keywords.items():
        for keyword in keywords:
            if keyword in text:
                detected_sector = sector
                break
        if detected_sector != "Unknown":
            break

    return {"Company": company, "Sector": detected_sector}


if __name__ == "__main__":

    files = [f for f in os.listdir(reports_folder) if f.endswith(".pdf")]

    print("Total reports:", len(files))

    with Pool(cpu_count()) as pool:
        results = pool.map(detect_sector, files)

    df = pd.DataFrame(results)

    # Save company-sector mapping
    df.to_csv("company_sector_mapping_brsr_25_26.csv", index=False)

    # Create sector-wise grouping
    sector_group = df.groupby("Sector")["Company"].apply(list).reset_index()

    sector_group["Companies"] = sector_group["Company"].apply(lambda x: ", ".join(x))

    sector_group = sector_group[["Sector", "Companies"]]

    sector_group.to_csv("sector_wise_companies_brsr_25_26.csv", index=False)

    print("Sector files created successfully")

In [ ]:
# ============================================================
# ESG / SECTION WORD COUNT VISUALIZATION
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD EXCEL FILE
# ============================================================

# CHANGE THIS TO YOUR FILE NAME
FILE_PATH = "/kaggle/working/detailed_word_counts_brsr_24_25.csv"

df = pd.read_csv(FILE_PATH)

print(df.head())

# ============================================================
# COLUMN ASSUMPTIONS
# ============================================================

# Column 1 -> Company Name
# Column 2 -> Section Name
# Column 3 -> Word Count

company_col = df.columns[0]
section_col = df.columns[1]
count_col = df.columns[2]

# ============================================================
# STACKED BAR CHART
# ============================================================

pivot_df = df.pivot_table(
    index=company_col,
    columns=section_col,
    values=count_col,
    aggfunc='sum',
    fill_value=0
)

plt.figure(figsize=(16,7))

pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(16,7)
)

plt.title("Section-wise Word Counts Across Companies")

plt.xlabel("Companies")
plt.ylabel("Word Count")

plt.xticks(rotation=90)

plt.tight_layout()

plt.savefig("stacked_bar_chart.png")

plt.show()

# ============================================================
# PIE CHART
# TOTAL SECTION DISTRIBUTION
# ============================================================

section_totals = df.groupby(section_col)[count_col].sum()

plt.figure(figsize=(8,8))

plt.pie(
    section_totals,
    labels=section_totals.index,
    autopct='%1.1f%%'
)

plt.title("Overall Section Distribution")

plt.savefig("section_distribution_pie.png")

plt.show()

# ============================================================
# HISTOGRAM
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(df[count_col], bins=15)

plt.title("Distribution of Word Counts")

plt.xlabel("Word Count")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig("wordcount_histogram.png")

plt.show()

# ============================================================
# DONE
# ============================================================

print("✅ Plots generated successfully!")